## Lab 4: Build a simple question-answering chatbot using pre-trained models

This notebook compares several pretrained QA models on shared questions.



### Models overview (quick facts)

- **DistilBERT uncased (SQuAD)**

  - Parameters: ~66M

  - Fine-tuning dataset: SQuAD v1.1

  - Pretraining data: BERT-base uncased (BooksCorpus + Wikipedia)

  - Creator: Hugging Face

  - Strength: lightweight English QA

  - Architecture: distilled encoder-only transformer (6 layers, 768 hidden)

- **DistilBERT cased (SQuAD)**

  - Parameters: ~66M

  - Fine-tuning dataset: SQuAD v1.1

  - Pretraining data: BERT-base cased (BooksCorpus + Wikipedia)

  - Creator: Hugging Face

  - Strength: English QA where casing helps entities/names

  - Architecture: distilled encoder-only transformer (6 layers, 768 hidden)

- Difference (uncased vs cased): uncased lowercases all text; cased preserves original casing (useful for names/abbreviations).

- **ALBERT base v2 (SQuAD2)**

  - Parameters: ~12M trainable (with sharing/factorized embeddings)

  - Fine-tuning dataset: SQuAD v2

  - Pretraining data: BooksCorpus + Wikipedia

  - Creator: Google Research

  - Strength: efficient QA with unanswerable handling

  - Architecture: encoder-only transformer with shared parameters (12 layers)

- **XLM-RoBERTa base (SQuAD2)**

  - Parameters: ~270M

  - Fine-tuning dataset: SQuAD v2

  - Pretraining data: CC-100 multilingual corpus

  - Creator: Meta AI

  - Strength: multilingual QA

  - Architecture: encoder-only transformer (12 layers, 768 hidden)


In [ ]:
!pip install transformers torch

In [ ]:
from transformers import pipeline

# Models to compare
models = [
    ("DistilBERT uncased (SQuAD)", "distilbert-base-uncased-distilled-squad"),
    ("DistilBERT cased (SQuAD)", "distilbert-base-cased-distilled-squad"),
    ("ALBERT base v2 (SQuAD2)", "twmkn9/albert-base-v2-squad2"),
    ("XLM-RoBERTa multilingual (SQuAD2)", "deepset/xlm-roberta-base-squad2"),
]

# Context for the models to answer questions about
context = """
Generative AI refers to models that create new content (text, images, audio, code) by learning patterns from large datasets. It works by predicting likely next tokens (or pixels/audio chunks) given a prompt. Examples include ChatGPT and Gemini for text, DALL·E and Stable Diffusion for images, and code assistants that draft snippets. Common uses: chatbots, summarization, drafting emails or reports, translation, design mockups, and generating code or test cases. A key benefit is faster content creation and productivity. A key risk is producing incorrect, biased, or fabricated information if prompts or data are weak.
"""

# Question set for side-by-side comparison
questions = [
    "What is generative AI?",
    "What is generative AI used for?",
    "What are some examples of generative AI?",
    "How does generative AI work?",
    "Name a benefit of generative AI.",
    "Name a risk of generative AI.",
]

def build_pipelines(model_items):
    # Build one QA pipeline per model with explicit tokenizer to avoid loading issues
    return [
        (
            label,
            pipeline(
                "question-answering",
                model=model_id,
                tokenizer=model_id,
            ),
        )
        for label, model_id in model_items
    ]

qa_pipelines = build_pipelines(models)

def ask_all(models_and_pipes, user_question, user_context):
    # Run one question across all models and print aligned outputs
    for label, qa in models_and_pipes:
        result = qa(question=user_question, context=user_context)
        print(f"  - {label}")
        print(f"    Answer: {result['answer']}")
        print(f"    Confidence: {result['score']:.4f}")
    print()

def ask_all_questions(models_and_pipes, user_questions, user_context):
    # Run the predefined question list across all models for easy comparison
    for idx, q in enumerate(user_questions, start=1):
        print("=" * 72)
        print(f"Q{idx}: {q}")
        print("-" * 72)
        ask_all(models_and_pipes, q, user_context)


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForQuestionAnswering LOAD REPORT from: twmkn9/albert-base-v2-squad2
Key                  | Status     |  | 
---------------------+------------+--+-
albert.pooler.bias   | UNEXPECTED |  | 
albert.pooler.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Could not extract SentencePiece model from C:\Users\vivek\.cache\huggingface\hub\models--twmkn9--albert-base-v2-squad2\snapshots\eb613ac0a88a507c701b46de597aff50a1089dc1\spiece.model using sentencepiece library due to 
SentencePieceExtractor requires the SentencePiece library but it was not found in your environment. Check out the instructions on the
installation page of its repo: https://github.com/google/sentencepiece#installation and follow the ones
that match your environment. Please note that you may need to restart your runtime after installation.
. Falling back to TikToken extractor.
Could not extract SentencePiece model from C:\U

TypeError: QuestionAnsweringPipeline.__init__() missing 1 required positional argument: 'tokenizer'

In [ ]:
# Compare the predefined set of questions across all models
ask_all_questions(qa_pipelines, questions, context)


In [ ]:
# Interactive loop: send each user question to every model (type 'exit' to stop)
while True:
    user_question = input("Ask a question (or type 'exit'): ")
    if user_question.lower() == "exit":
        print("Chatbot exited.")
        break
    for label, qa in qa_pipelines:
        response = qa(question=user_question, context=context)
        print(f"[{label}] Answer: {response['answer']} (conf: {response['score']:.4f})")
    print()
